# 01 — Data Audit

**Goal:** understand exactly what StatsBomb open data is available for Barcelona La Liga matches, and identify which season–opponent pairs give us both a home *and* an away fixture — the paired comparisons that will be central to the analysis.

This notebook only loads and inspects data. No metrics, no modelling.

In [ ]:
import pandas as pd
from statsbombpy import sb

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)

## 1. Which competitions does StatsBomb open data cover?

StatsBomb free tier covers a limited set of competitions. La Liga has `competition_id = 11`.

In [ ]:
competitions = sb.competitions()
competitions[competitions['competition_name'] == 'La Liga'][['competition_id', 'season_id', 'season_name']]

## 2. Pull all La Liga matches and keep Barcelona fixtures

We loop over every available La Liga season, pull the match list, and filter to matches where Barcelona appeared (home or away).

In [ ]:
COMPETITION_ID = 11  # La Liga
BARCA_NAME = 'Barcelona'

la_liga_seasons = (
    competitions[competitions['competition_name'] == 'La Liga']
    [['season_id', 'season_name']]
    .drop_duplicates()
)

all_matches = []
for _, row in la_liga_seasons.iterrows():
    season_matches = sb.matches(competition_id=COMPETITION_ID, season_id=row['season_id'])
    season_matches['season_name'] = row['season_name']
    all_matches.append(season_matches)

matches = pd.concat(all_matches, ignore_index=True)

barca = matches[
    (matches['home_team'] == BARCA_NAME) | (matches['away_team'] == BARCA_NAME)
].copy()

print(f"Total La Liga matches in open data: {len(matches)}")
print(f"Barcelona fixtures: {len(barca)}")

## 3. Label each fixture: Barça home or away? Who is the opponent?

We add two columns: `barca_venue` ('home' or 'away') and `opponent` (the other team).

In [ ]:
barca['barca_venue'] = barca['home_team'].apply(
    lambda t: 'home' if t == BARCA_NAME else 'away'
)
barca['opponent'] = barca.apply(
    lambda r: r['away_team'] if r['home_team'] == BARCA_NAME else r['home_team'],
    axis=1
)

barca[['season_name', 'match_date', 'barca_venue', 'opponent',
       'home_score', 'away_score']].head(10)

## 4. Find paired fixtures

The core of the analysis relies on comparing Barça's behaviour home vs away **against the same opponent in the same season**. Here we identify every (season, opponent) pair where we have both a home and an away fixture.

In [ ]:
# Count how many times each (season, opponent) appears in each venue
venue_counts = (
    barca.groupby(['season_name', 'opponent', 'barca_venue'])
    .size()
    .unstack('barca_venue', fill_value=0)
    .reset_index()
)

# A valid pair needs exactly 1 home and 1 away fixture
paired = venue_counts[
    (venue_counts.get('home', 0) == 1) & (venue_counts.get('away', 0) == 1)
]

print(f"Paired (season, opponent) combinations: {len(paired)}")
print(f"Seasons covered: {paired['season_name'].nunique()}")
print(f"Unique opponents covered: {paired['opponent'].nunique()}")
paired.head(10)

## 5. Coverage summary

How many paired fixtures per season? This tells us how much data we have for the home/away comparison.

In [ ]:
paired.groupby('season_name').size().rename('paired_fixtures').sort_index()

## What we now know

- How many Barça La Liga matches are in the open data.
- Which seasons are covered.
- How many (season, opponent) pairs give us a home **and** an away fixture — these are the observations for the core analysis.

**Next notebook (`02_feature_engineering.ipynb`):** for each of these matches, compute the first tactical aggression dimension — pressing intensity via PPDA.